In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [2]:
fecha_mes_base='2026-05-01'
tipi_cond1='DINERS TC'
tipi_cond2='xx'
tipi_cond3='xx'
tb_tipolofia='tTipologia_Diners_TC'
servidor_01=21
tipi_cod='cod'
tipi_resp_cod='H'
tipi_descrip='[NIVEL 4]'
tipi_estado='[NIVEL 2]'
tipi_resp_estado='NO CONTACTO'
tipi_subdescripcion='[NIVEL 3]'
tnum_tb='tNumeroDinersTc'
tnum_dni='NUMERO_DOCUMENTO'
tlista_generada='borrar_tc_dinner'
get_base=since_base_maestra_tc_dinners

# df_vicidial=since_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado,tnum_tb,tnum_dni)


In [3]:
df_vicidial=since_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado,tnum_tb,tnum_dni)


In [4]:

df_vicidial.filter(F.col('phone_number')=='959643856').orderBy(F.col('fecha_hora_llamada').desc()).show()

+------+------------+----------------+--------------------+--------------+-------------+--------------------+-----------------+-------------------+--------+--------------------+-----------------+-----------------+------------+-----------+-------------+-----+--------+--------------------+----+-----------------+------------------+-----------+----------------+--------------+---------------+---------------+--------------------+----------+-----------------+---------------+--------------+----------------+
|codigo|phone_number|vendor_lead_code|         dial_method|numero_campana|dni_ejecutivo|           ejecutivo|   nombre_campana| fecha_hora_llamada|duracion|         call_result| list_description|        list_name|fecha_agenda|comentarios|fecha_llamada|tramo|  hora_a|         descripcion|peso|n_mejor_resul_cli|n_mejor_resul_telf|n_ult_resul|mejor_codigo_cli|ult_codigo_cli|ult_call_result|fecha_llamada_1|fecha_hora_llamada_1|q_intentos|mejor_codigo_telf|q_intentos_telf|q_intentos_dia|q_intent

In [52]:


from sqlalchemy import create_engine

engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [ ]:
select distinct codigo,id_banco from crm_target.alfcc_acciones
limit 10

In [71]:
fecha_ref='2026-05-08'
query = f"""
select distinct respuesta,agente from crm_target.valentina_llamadas
WHERE app=17
and fecha_llamada='2026-05-08'
limit 10

    """

df_vicidial= pd.read_sql(query, engine_mysql)

In [72]:
df_vicidial.head()


,respuesta,agente
0,Answering Machine Auto,VDAD
1,No Answer AutoDial,VDAD
2,Busy Auto,VDAD
3,Agent Not Available,VDAD
4,Disconnected Number Auto,VDAD


In [42]:


df_vicidial=df_vicidial.withColumn(
        "vendor_lead_code",
        F.right(
            F.concat(F.lit("00000000"), F.col("vendor_lead_code")),
            F.lit(8)
        )
    )

query = f"""
    select 
    distinct
    Telefonos as phone_number,{tnum_dni} as vendor_lead_code_1
    from DANTALION.dbo.{tnum_tb}
    """
df_tnumer=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_tnumer = df_tnumer.withColumn(
        "vendor_lead_code_1",
        F.right(
            F.concat(F.lit("00000000"), F.col("vendor_lead_code_1")),
            F.lit(8)
        )
    )

df_vicidial=df_vicidial.join(df_tnumer,['phone_number'],'left')

df_vicidial = df_vicidial.withColumn(
    "vendor_lead_code",
    F.coalesce(F.col("vendor_lead_code"), F.col("vendor_lead_code_1"))
).drop("vendor_lead_code_1")
df_vicidial=df_vicidial.filter(F.col('vendor_lead_code').isNotNull())

df_vicidial = (
    df_vicidial
    .withColumn("fecha_llamada", F.to_date(F.col("fecha_hora_llamada")))
    .withColumn("tramo", F.hour(F.col("fecha_hora_llamada")))
    .withColumn("hora_a", F.date_format(F.col("fecha_hora_llamada"), "HH:mm:ss"))
)

query = f"""
    SELECT {tipi_cod} as codigo
    , case
        when {tipi_cod}='CALLBK' then 'VOLVER A LLAMAR - call'
        else {tipi_descrip} 
    end as descripcion
    ,case 
        when {tipi_cod}='CALLBK' then 1200
        else peso 
    end as peso  FROM [ODIN].[dbo].{tb_tipolofia}
    where LEFT({tipi_cod},1)='{tipi_resp_cod}' or {tipi_estado}='{tipi_resp_estado}' or {tipi_cod}='CALLBK'
    """
df_tipi=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

df_vicidial=df_vicidial.join(df_tipi,["codigo"],"left")

df_vicidial = df_vicidial.withColumn(
    'peso',
    F.when(F.col('codigo') == 'INCALL', 100000)
    .when((F.col('codigo') == 'DCMX') & (F.col('duracion') > 15), 100001)
    .when(F.col('codigo') == 'DCMX', 100002)
    .otherwise(F.col('peso'))
)

window_spec = (Window.partitionBy("vendor_lead_code")
                .orderBy(col("peso").asc_nulls_last()
                ))
df_vicidial = df_vicidial.withColumn("n_mejor_resul_cli", row_number().over(window_spec))

window_spec = (Window.partitionBy("vendor_lead_code",'phone_number')
                .orderBy(
                    col("peso").asc_nulls_last(),
                    col("hora_a").desc_nulls_last()
                ))

df_vicidial = df_vicidial.withColumn(
    "n_mejor_resul_telf",
    F.when(
        F.col("duracion") > 15,
        F.row_number().over(window_spec)
    )
)

window_spec = (Window.partitionBy("vendor_lead_code")
                .orderBy(
                    col("fecha_hora_llamada").desc_nulls_last()
                ))
df_vicidial = df_vicidial.withColumn("n_ult_resul", row_number().over(window_spec))

window_part = Window.partitionBy("vendor_lead_code")
df_vicidial = df_vicidial.withColumn(
    "mejor_codigo_cli",
    F.max(
        F.when(F.col("n_mejor_resul_cli") == 1, F.col("codigo"))
    ).over(window_part)
)

df_vicidial = df_vicidial.withColumn(
    "ult_codigo_cli",
    F.max(
        F.when(F.col("n_ult_resul") == 1, F.col("codigo"))
    ).over(window_part)
)

df_vicidial = df_vicidial.withColumn(
    "ult_call_result",
    F.max(
        F.when(F.col("n_ult_resul") == 1, F.col("call_result"))
    ).over(window_part)
)

df_vicidial = df_vicidial.withColumn(
    "fecha_llamada_1",
    F.max(
        F.when(F.col("n_ult_resul") == 1, F.col("fecha_llamada"))
    ).over(window_part)
)

df_vicidial = df_vicidial.withColumn(
    "fecha_hora_llamada_1",
    F.max(
        F.when(F.col("n_ult_resul") == 1, F.col("fecha_hora_llamada"))
    ).over(window_part)
)  

df_vicidial = df_vicidial.withColumn("q_intentos",count("*").over(window_part))

window_part = Window.partitionBy("vendor_lead_code",'phone_number')
df_vicidial = df_vicidial.withColumn(
    "mejor_codigo_telf",
    F.max(
        F.when(F.col("n_mejor_resul_telf") == 1, F.col("codigo"))
    ).over(window_part)
)

df_vicidial = df_vicidial.withColumn("q_intentos_telf",count("*").over(window_part))

window_part = Window.partitionBy('fecha_llamada',"vendor_lead_code")
df_vicidial = df_vicidial.withColumn("q_intentos_dia",count("*").over(window_part))

window_part = Window.partitionBy("vendor_lead_code")
df_vicidial = df_vicidial.withColumn(
    "q_intentos_dia_1",
    F.max(
        F.when(F.col("fecha_llamada") == F.col("fecha_llamada_1"), F.col("q_intentos_dia"))
    ).over(window_part)
)

return df_vicidial


AttributeError: 'DataFrame' object has no attribute 'withColumn'

In [4]:

query = """
SELECT
    distinct term_reason
FROM
	valentina_llamadas
WHERE
	fecha_llamada = '2026-05-08'
	AND app = 12
limit 10
"""

df_dni = pd.read_sql(query, engine_mysql)
df_dni.head()

,term_reason
0,CALLER
1,NONE
2,QUEUETIMEOUT
3,ABANDON


In [ ]:
['id', 'telefono', 'fecha_llamada', 'hora_llamada', 'hora_colgado', 'duracion', 'tipificacion', 'respuesta', 'agente', 'cid', 'app', 'campana', 'id_lista', 'servidor', 'estado', 'hora_registro', 'uniqueid', 'lead_id', 'term_reason']

['id', 'telefono', 'fecha_llamada', 'hora_llamada', 'hora_colgado', 'duracion', 'tipificacion', 'respuesta', 'agente', 'cid', 'app', 'campana', 'id_lista', 'servidor', 'estado', 'hora_registro', 'uniqueid', 'lead_id', 'term_reason']


In [8]:
df_dni.head()

,id,telefono,fecha_llamada,hora_llamada,hora_colgado,duracion,tipificacion,respuesta,agente,cid,app,campana,id_lista,servidor,estado,hora_registro,uniqueid,lead_id,term_reason
0,20109694,960404219,2026-05-08,0 days 09:00:58,0 days 09:00:58,0,AA,Answering Machine Auto,VDAD,3292324,12,ALFIN,122026050710,valentina1.target.mastermold.dev,0,None,1778248839.1121,12427364,CALLER
1,20109695,954714252,2026-05-08,0 days 09:00:39,0 days 09:00:39,0,AB,Busy Auto,VDAD,3283678,12,ALFIN,122026050710,valentina1.target.mastermold.dev,0,None,1778248839.1123,12434114,NONE
2,20109696,992810375,2026-05-08,0 days 09:00:52,0 days 09:00:52,0,AA,Answering Machine Auto,VDAD,3263973,12,ALFIN,122026050710,valentina1.target.mastermold.dev,0,None,1778248839.1124,12427353,CALLER
3,20109697,933808730,2026-05-08,0 days 09:00:55,0 days 09:00:55,0,AA,Answering Machine Auto,VDAD,3300860,12,ALFIN,122026050710,valentina1.target.mastermold.dev,0,None,1778248839.1126,12430126,CALLER
4,20109698,999599684,2026-05-08,0 days 09:00:51,0 days 09:00:51,0,AA,Answering Machine Auto,VDAD,3302694,12,ALFIN,122026050710,valentina1.target.mastermold.dev,0,None,1778248839.1133,12430892,CALLER


In [ ]:

query = """
Select 
NUMDOC,FECHA_LLAMADA,HORA_LLAMADA,HORA_TERMINO,TELEFONO,CODTIPIF,IDASESOR,TMO,TIPO,"17" AS SERVICIO
from alfcc_feedback 
where fecha_llamada=CURDATE()- INTERVAL 0 DAY
"""

df_dni = pd.read_sql(query, engine_mysql)

df_long = df_dni.melt(
    id_vars='NUMERO_DOCUMENTO',
    value_vars=cols_tel,
    var_name='tipo_telf',
    value_name='CELULAR'
)
df_long['CELULAR'] = (
    df_long['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_long = df_long[
    (df_long['CELULAR'].notna()) &
    (df_long['CELULAR'] != '') &
    (df_long['CELULAR'].str.len() == 9) &
    (df_long['CELULAR'].str.startswith('9'))
]

